In [1]:
!pip install pyspark


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("df") \
    .getOrCreate()

1. Load the Data in PySpark

In [3]:
from google.colab import files

uploaded = files.upload()

Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv


In [4]:
df = spark.read.csv(
    "manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [5]:
df.show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [6]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [7]:
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [8]:
df.count()

120000

In [9]:
df.select("production_id", "plant_id", "machine_id", "product_family", "units_produced").show()

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

In [10]:


df.dtypes

[('production_id', 'string'),
 ('production_date', 'date'),
 ('plant_id', 'string'),
 ('machine_id', 'string'),
 ('operator_id', 'string'),
 ('shift', 'string'),
 ('product_code', 'string'),
 ('product_family', 'string'),
 ('units_planned', 'int'),
 ('units_produced', 'int'),
 ('defective_units', 'int'),
 ('downtime_minutes', 'int'),
 ('energy_kwh', 'double'),
 ('temperature_c', 'double'),
 ('vibration_mm_s', 'double'),
 ('quality_status', 'string'),
 ('maintenance_flag', 'string'),
 ('supervisor', 'string')]

In [11]:
df.sample(fraction=0.0001, seed=42).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0048985|     2026-01-12|    PL05|     MC022|     OP0143|Morning|     PMP-310|         Pumps|          584|           558|             22|              55|    214.88|        58.09|          2.46|          Pass|              No|     Farah|
|    PR0054858|     

2. Creating DataFrames

In [12]:
plant_data = [
    ("PL01", "Chennai Motors Plant", "Chennai"),
    ("PL02", "Pune Controls Plant", "Pune"),
    ("PL03", "Hosur Pumps Plant", "Hosur"),
    ("PL04", "Coimbatore Assembly Plant", "Coimbatore"),
    ("PL05", "Bangalore Electronics Plant", "Bangalore"),
    ("PL06", "Hyderabad Components Plant", "Hyderabad")
]

In [21]:
plant_df = spark.createDataFrame(
    plant_data,
    ["plant_id", "plant_name", "city"]
)

plant_df.show()

+--------+--------------------+----------+
|plant_id|          plant_name|      city|
+--------+--------------------+----------+
|    PL01|Chennai Motors Plant|   Chennai|
|    PL02| Pune Controls Plant|      Pune|
|    PL03|   Hosur Pumps Plant|     Hosur|
|    PL04|Coimbatore Assemb...|Coimbatore|
|    PL05|Bangalore Electro...| Bangalore|
|    PL06|Hyderabad Compone...| Hyderabad|
+--------+--------------------+----------+



In [18]:
from pyspark.sql.types import StructType, StructField, StringType

plant_schema = StructType([
    StructField("plant_id", StringType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])

plant_df2 = spark.createDataFrame(plant_data, plant_schema)

plant_df2.show()

+--------+--------------------+----------+
|plant_id|          plant_name|      city|
+--------+--------------------+----------+
|    PL01|Chennai Motors Plant|   Chennai|
|    PL02| Pune Controls Plant|      Pune|
|    PL03|   Hosur Pumps Plant|     Hosur|
|    PL04|Coimbatore Assemb...|Coimbatore|
|    PL05|Bangalore Electro...| Bangalore|
|    PL06|Hyderabad Compone...| Hyderabad|
+--------+--------------------+----------+



In [15]:
plant_df.printSchema()
plant_df2.printSchema()

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [17]:
from pyspark.sql.functions import lit

plant_df = plant_df.withColumn("country", lit("India"))

plant_df.show()

+--------+--------------------+----------+-------+
|plant_id|          plant_name|      city|country|
+--------+--------------------+----------+-------+
|    PL01|Chennai Motors Plant|   Chennai|  India|
|    PL02| Pune Controls Plant|      Pune|  India|
|    PL03|   Hosur Pumps Plant|     Hosur|  India|
|    PL04|Coimbatore Assemb...|Coimbatore|  India|
|    PL05|Bangalore Electro...| Bangalore|  India|
|    PL06|Hyderabad Compone...| Hyderabad|  India|
+--------+--------------------+----------+-------+



3.Pandas DataFrame


In [22]:
import pandas as pd

pdf = pd.read_csv("manufacturing_production_120k.csv")

In [23]:
pdf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [24]:
pdf.tail()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
119995,PR0119996,2026-07-13,PL05,MC022,OP0193,Evening,MTR-200,Motors,435,399,1,73.0,126.07,62.79,5.73,Rework,No,Bharat
119996,PR0119997,2026-04-26,PL05,MC024,OP0124,Evening,CTL-130,Controllers,813,777,8,49.0,318.22,63.05,3.53,Pass,No,Irfan
119997,PR0119998,2026-02-01,PL06,MC027,OP0051,Evening,VLV-120,Valves,650,597,13,0.0,269.46,60.52,3.81,Pass,No,Bharat
119998,PR0119999,2026-01-17,PL01,MC001,OP0452,Evening,MTR-300,Motors,887,899,22,21.0,184.62,64.89,5.20,Pass,Yes,Farah
119999,PR0120000,2026-08-16,PL02,MC007,OP0420,Morning,MTR-200,Motors,547,477,4,43.0,107.95,62.50,3.97,Pass,No,Farah


In [25]:
pdf.shape

(120000, 18)

In [26]:
pdf.columns

Index(['production_id', 'production_date', 'plant_id', 'machine_id',
       'operator_id', 'shift', 'product_code', 'product_family',
       'units_planned', 'units_produced', 'defective_units',
       'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s',
       'quality_status', 'maintenance_flag', 'supervisor'],
      dtype='object')

In [27]:
pdf.dtypes

,0
production_id,object
production_date,object
plant_id,object
machine_id,object
operator_id,object
shift,object
product_code,object
product_family,object
units_planned,int64
units_produced,int64


In [28]:
pdf[["plant_id", "machine_id", "units_produced"]]

,plant_id,machine_id,units_produced
0,PL03,MC011,993
1,PL04,MC019,836
2,PL01,MC002,380
3,PL03,MC013,720
4,PL04,MC020,296
...,...,...,...
119995,PL05,MC022,399
119996,PL05,MC024,777
119997,PL06,MC027,597
119998,PL01,MC001,899


In [29]:
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})
pdf_renamed.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [30]:
pdf[pdf["quality_status"] == "Reject"]

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862,895,51,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532,527,26,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623,560,23,14.0,217.69,55.23,3.28,Reject,Yes,Farah
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119979,PR0119980,2026-07-21,PL02,MC006,OP0591,Morning,VLV-320,Valves,1149,1036,48,40.0,316.55,70.09,2.90,Reject,No,Harini
119988,PR0119989,2026-01-29,PL01,MC003,OP0021,Night,PMP-310,Pumps,1195,1053,26,72.0,240.94,62.27,4.93,Reject,No,Farah
119989,PR0119990,2026-07-20,PL06,MC027,OP0217,Evening,CTL-230,Controllers,782,796,10,71.0,150.47,66.59,4.08,Reject,No,Bharat
119991,PR0119992,2026-04-07,PL05,MC024,OP0186,Morning,PMP-310,Pumps,1187,1024,35,32.0,486.71,54.09,3.37,Reject,No,Farah


In [31]:
pdf[pdf["temperature_c"] > 75]

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
14,PR0000015,2026-03-15,PL02,MC007,OP0238,Morning,MTR-300,Motors,1086,1042,4,172.0,239.64,76.39,2.81,Pass,No,Deepa
16,PR0000017,2026-08-04,PL03,MC011,OP0533,Morning,CTL-330,Controllers,1152,963,28,91.0,347.34,75.09,4.00,Pass,Yes,Gopal
19,PR0000020,2026-02-18,PL03,MC014,OP0563,Morning,PMP-310,Pumps,646,573,13,64.0,277.73,76.92,2.35,Pass,No,Irfan
26,PR0000027,2026-01-03,PL05,MC023,OP0190,Morning,PMP-210,Pumps,358,310,6,46.0,78.29,75.42,3.72,Rework,No,Anita
54,PR0000055,2026-07-31,PL01,MC005,OP0212,Morning,CTL-230,Controllers,499,478,10,32.0,112.60,75.10,3.38,Pass,No,Farah
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119767,PR0119768,2026-01-29,PL02,MC007,OP0262,Evening,CTL-130,Controllers,1129,1139,0,30.0,362.12,77.17,1.98,Pass,No,Anita
119810,PR0119811,2026-07-09,PL05,MC024,OP0372,Evening,CTL-230,Controllers,819,687,45,50.0,171.84,75.24,4.17,Reject,No,Bharat
119833,PR0119834,2026-06-22,PL04,MC020,OP0038,Morning,CTL-330,Controllers,749,669,12,46.0,291.22,80.53,3.05,Pass,No,Anita
119848,PR0119849,2026-02-25,PL06,MC027,OP0563,Morning,PMP-310,Pumps,925,816,28,0.0,258.03,75.94,3.53,Rework,No,Charan


In [32]:
pdf[(pdf["shift"] == "Night") & (pdf["plant_id"] == "PL06")]

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
12,PR0000013,2026-07-19,PL06,MC027,OP0172,Night,CTL-330,Controllers,1004,867,27,66.0,388.61,50.88,5.70,Pass,Yes,Deepa
31,PR0000032,2026-05-28,PL06,MC030,OP0074,Night,CTL-230,Controllers,819,773,11,27.0,354.30,63.51,3.18,Pass,No,Deepa
55,PR0000056,2026-09-01,PL06,MC029,OP0239,Night,VLV-320,Valves,1102,970,62,55.0,450.64,79.15,3.88,Rework,Yes,Deepa
59,PR0000060,2026-06-21,PL06,MC026,OP0077,Night,PMP-210,Pumps,917,927,21,2.0,437.84,64.99,5.11,Pass,No,Bharat
61,PR0000062,2026-07-11,PL06,MC028,OP0352,Night,CTL-230,Controllers,698,639,0,17.0,205.25,74.27,3.37,Pass,No,Anita
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119926,PR0119927,2026-01-18,PL06,MC030,OP0219,Night,MTR-100,Motors,290,295,2,37.0,98.08,48.91,2.89,Reject,No,Charan
119953,PR0119954,2026-09-17,PL06,MC028,OP0481,Night,CTL-330,Controllers,1047,1021,38,29.0,469.84,63.81,5.56,Pass,No,Bharat
119958,PR0119959,2026-07-27,PL06,MC029,OP0243,Night,MTR-100,Motors,1139,1108,6,51.0,395.18,60.16,2.40,Pass,No,Bharat
119970,PR0119971,2026-09-13,PL06,MC030,OP0112,Night,PMP-210,Pumps,890,889,18,19.0,218.48,66.43,3.58,Reject,Yes,Deepa


4.Dask DataFrame

In [33]:
!pip install "dask[dataframe]"

In [34]:
import dask.dataframe as dd

ddf = dd.read_csv("manufacturing_production_120k.csv")

In [36]:
import dask.dataframe as dd

ddf = dd.read_csv(
    "manufacturing_production_120k.csv",
    dtype={"downtime_minutes": "float64"}
)

ddf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [37]:
ddf = dd.read_csv(
    "manufacturing_production_120k.csv",
    assume_missing=True
)

ddf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060.0,993.0,30.0,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809.0,836.0,0.0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442.0,380.0,17.0,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787.0,720.0,7.0,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344.0,296.0,7.0,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [38]:
ddf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060.0,993.0,30.0,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809.0,836.0,0.0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442.0,380.0,17.0,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787.0,720.0,7.0,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344.0,296.0,7.0,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [39]:

ddf.npartitions

1

In [40]:
ddf = dd.read_csv("manufacturing_production_120k.csv", blocksize="2MB")
ddf.npartitions

6

In [42]:
import dask.dataframe as dd

ddf = dd.read_csv(
    "manufacturing_production_120k.csv",
    assume_missing=True
)

In [43]:
selected = ddf[["plant_id", "product_family", "units_produced"]]
selected.head()

,plant_id,product_family,units_produced
0,PL03,Motors,993.0
1,PL04,Controllers,836.0
2,PL01,Motors,380.0
3,PL03,Pumps,720.0
4,PL04,Pumps,296.0


In [44]:
rejected = ddf[ddf["quality_status"] == "Reject"]
rejected.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809.0,836.0,0.0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442.0,380.0,17.0,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862.0,895.0,51.0,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532.0,527.0,26.0,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623.0,560.0,23.0,14.0,217.69,55.23,3.28,Reject,Yes,Farah


In [45]:
avg_units = ddf.groupby("plant_id")["units_produced"].mean()
avg_units

Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 5 expressions
Expr=((ArrowStringConversion(frame=FromMapProjectable(7f16c1f))[['plant_id', 'units_produced']]).mean(observed=False, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='units_produced'))['units_produced']

In [46]:
defects = ddf.groupby("product_family")["defective_units"].sum()
defects

Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: sum, 4 expressions
Expr=Sum(frame=ArrowStringConversion(frame=FromMapProjectable(7f16c1f))[['product_family', 'defective_units']], observed=False, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='defective_units')

In [47]:
print(avg_units.compute())
print(defects.compute())
print(len(rejected))

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64
product_family
Controllers    507781.0
Motors         505287.0
Pumps          507864.0
Valves         509312.0
Name: defective_units, dtype: float64
22744


In [49]:
import pandas as pd
import time

pdf = pd.read_csv("manufacturing_production_120k.csv")
start = time.time()
pandas_result = pdf.groupby("product_family")["defective_units"].sum()
print("Pandas:", round(time.time() - start, 4), "seconds")
print(pandas_result)

start = time.time()
dask_result = ddf.groupby("product_family")["defective_units"].sum().compute()
print("Dask:", round(time.time() - start, 4), "seconds")
print(dask_result)

Pandas: 0.0162 seconds
product_family
Controllers    507781
Motors         505287
Pumps          507864
Valves         509312
Name: defective_units, dtype: int64
Dask: 0.7404 seconds
product_family
Controllers    507781.0
Motors         505287.0
Pumps          507864.0
Valves         509312.0
Name: defective_units, dtype: float64


5.Manipulating Df


In [50]:
from pyspark.sql import functions as F

df = spark.read.csv(
    "manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [52]:
df_mfg = df.withColumn(
    "production_efficiency",
    F.round(F.col("units_produced") / F.col("units_planned") * 100, 2)
)

df_mfg.select("production_id", "units_planned", "units_produced", "production_efficiency").show()

+-------------+-------------+--------------+---------------------+
|production_id|units_planned|units_produced|production_efficiency|
+-------------+-------------+--------------+---------------------+
|    PR0000001|         1060|           993|                93.68|
|    PR0000002|          809|           836|               103.34|
|    PR0000003|          442|           380|                85.97|
|    PR0000004|          787|           720|                91.49|
|    PR0000005|          344|           296|                86.05|
|    PR0000006|          958|           795|                82.99|
|    PR0000007|         1165|           955|                81.97|
|    PR0000008|          862|           895|               103.83|
|    PR0000009|          313|           289|                92.33|
|    PR0000010|          532|           527|                99.06|
|    PR0000011|          577|           512|                88.73|
|    PR0000012|          980|           968|                98

In [53]:
df_mfg = df_mfg.withColumn(
    "good_units",
    F.col("units_produced") - F.col("defective_units")
)

df_mfg.select("production_id", "units_produced", "defective_units", "good_units").show()

+-------------+--------------+---------------+----------+
|production_id|units_produced|defective_units|good_units|
+-------------+--------------+---------------+----------+
|    PR0000001|           993|             30|       963|
|    PR0000002|           836|              0|       836|
|    PR0000003|           380|             17|       363|
|    PR0000004|           720|              7|       713|
|    PR0000005|           296|              7|       289|
|    PR0000006|           795|             50|       745|
|    PR0000007|           955|             10|       945|
|    PR0000008|           895|             51|       844|
|    PR0000009|           289|              8|       281|
|    PR0000010|           527|             26|       501|
|    PR0000011|           512|             14|       498|
|    PR0000012|           968|             23|       945|
|    PR0000013|           867|             27|       840|
|    PR0000014|           456|              6|       450|
|    PR0000015

In [54]:
df_mfg = df_mfg.withColumn("country", F.lit("India"))

df_mfg.select("production_id", "country").show()

+-------------+-------+
|production_id|country|
+-------------+-------+
|    PR0000001|  India|
|    PR0000002|  India|
|    PR0000003|  India|
|    PR0000004|  India|
|    PR0000005|  India|
|    PR0000006|  India|
|    PR0000007|  India|
|    PR0000008|  India|
|    PR0000009|  India|
|    PR0000010|  India|
|    PR0000011|  India|
|    PR0000012|  India|
|    PR0000013|  India|
|    PR0000014|  India|
|    PR0000015|  India|
|    PR0000016|  India|
|    PR0000017|  India|
|    PR0000018|  India|
|    PR0000019|  India|
|    PR0000020|  India|
+-------------+-------+
only showing top 20 rows


In [56]:
df_mfg = df_mfg.withColumnRenamed("energy_kwh", "energy_consumed")

df_mfg.select("production_id", "energy_consumed").show()

+-------------+---------------+
|production_id|energy_consumed|
+-------------+---------------+
|    PR0000001|         388.11|
|    PR0000002|         398.58|
|    PR0000003|         183.34|
|    PR0000004|         217.61|
|    PR0000005|         115.47|
|    PR0000006|         290.26|
|    PR0000007|         355.56|
|    PR0000008|         266.06|
|    PR0000009|          87.62|
|    PR0000010|         112.17|
|    PR0000011|          127.0|
|    PR0000012|         377.97|
|    PR0000013|         388.61|
|    PR0000014|         108.83|
|    PR0000015|         239.64|
|    PR0000016|         289.55|
|    PR0000017|         347.34|
|    PR0000018|         241.05|
|    PR0000019|         137.99|
|    PR0000020|         277.73|
+-------------+---------------+
only showing top 20 rows


In [57]:
df_mfg = df_mfg.withColumn(
    "units_produced",
    F.col("units_produced").cast("double")
)

df_mfg.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: double (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_consumed: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)
 |-- production_efficiency: double (nullable = true)
 |-- good_units: integer (nullable = true)
 |-- country: string (nullable = false)



In [58]:
df_mfg = df_mfg.withColumn(
    "defect_percentage",
    F.round(F.col("defective_units") / F.col("units_produced") * 100, 2)
)

df_mfg.select("production_id", "defective_units", "units_produced", "defect_percentage").show()

+-------------+---------------+--------------+-----------------+
|production_id|defective_units|units_produced|defect_percentage|
+-------------+---------------+--------------+-----------------+
|    PR0000001|             30|         993.0|             3.02|
|    PR0000002|              0|         836.0|              0.0|
|    PR0000003|             17|         380.0|             4.47|
|    PR0000004|              7|         720.0|             0.97|
|    PR0000005|              7|         296.0|             2.36|
|    PR0000006|             50|         795.0|             6.29|
|    PR0000007|             10|         955.0|             1.05|
|    PR0000008|             51|         895.0|              5.7|
|    PR0000009|              8|         289.0|             2.77|
|    PR0000010|             26|         527.0|             4.93|
|    PR0000011|             14|         512.0|             2.73|
|    PR0000012|             23|         968.0|             2.38|
|    PR0000013|          

In [59]:
df_mfg = df_mfg.withColumn(
    "energy_per_unit",
    F.round(F.col("energy_consumed") / F.col("units_produced"), 4)
)

df_mfg.select("production_id", "energy_consumed", "units_produced", "energy_per_unit").show()

+-------------+---------------+--------------+---------------+
|production_id|energy_consumed|units_produced|energy_per_unit|
+-------------+---------------+--------------+---------------+
|    PR0000001|         388.11|         993.0|         0.3908|
|    PR0000002|         398.58|         836.0|         0.4768|
|    PR0000003|         183.34|         380.0|         0.4825|
|    PR0000004|         217.61|         720.0|         0.3022|
|    PR0000005|         115.47|         296.0|         0.3901|
|    PR0000006|         290.26|         795.0|         0.3651|
|    PR0000007|         355.56|         955.0|         0.3723|
|    PR0000008|         266.06|         895.0|         0.2973|
|    PR0000009|          87.62|         289.0|         0.3032|
|    PR0000010|         112.17|         527.0|         0.2128|
|    PR0000011|          127.0|         512.0|          0.248|
|    PR0000012|         377.97|         968.0|         0.3905|
|    PR0000013|         388.61|         867.0|         

In [61]:
df_mfg.select(
    "production_id",
    "production_efficiency",
    "good_units",
    "country",
    "energy_consumed",
    "units_produced",
    "defect_percentage",
    "energy_per_unit"
).show()

+-------------+---------------------+----------+-------+---------------+--------------+-----------------+---------------+
|production_id|production_efficiency|good_units|country|energy_consumed|units_produced|defect_percentage|energy_per_unit|
+-------------+---------------------+----------+-------+---------------+--------------+-----------------+---------------+
|    PR0000001|                93.68|       963|  India|         388.11|         993.0|             3.02|         0.3908|
|    PR0000002|               103.34|       836|  India|         398.58|         836.0|              0.0|         0.4768|
|    PR0000003|                85.97|       363|  India|         183.34|         380.0|             4.47|         0.4825|
|    PR0000004|                91.49|       713|  India|         217.61|         720.0|             0.97|         0.3022|
|    PR0000005|                86.05|       289|  India|         115.47|         296.0|             2.36|         0.3901|
|    PR0000006|         

6.Dropping Data


In [62]:
df_q1 = df.drop("supervisor")
df_q1.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)



In [63]:
df_q2 = df.drop("supervisor", "maintenance_flag")
print(df_q2.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status']


In [64]:
df_q3 = df.dropDuplicates()
df_q3.count()

120000

In [66]:
df_q4 = df.dropDuplicates(["production_id"])
df_q4.count()

120000

In [67]:
df_q5 = df.dropna(subset=["operator_id"])
df_q5.count()

118476

In [ ]:
df_q6 = df.dropna(how="all", subset=["temperature_c", "vibration_mm_s"])
df_q6.count()

In [68]:
df.select("product_family").distinct().show()

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+



In [70]:
df.select("plant_id", "machine_id").distinct().orderBy("plant_id", "machine_id").show()

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC002|
|    PL01|     MC003|
|    PL01|     MC004|
|    PL01|     MC005|
|    PL02|     MC006|
|    PL02|     MC007|
|    PL02|     MC008|
|    PL02|     MC009|
|    PL02|     MC010|
|    PL03|     MC011|
|    PL03|     MC012|
|    PL03|     MC013|
|    PL03|     MC014|
|    PL03|     MC015|
|    PL04|     MC016|
|    PL04|     MC017|
|    PL04|     MC018|
|    PL04|     MC019|
|    PL04|     MC020|
+--------+----------+
only showing top 20 rows


7.SORTING

In [71]:
from pyspark.sql import functions as F

In [72]:
df.orderBy(F.col("units_produced").asc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [73]:
df.orderBy(F.col("units_produced").desc()).show()


+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [74]:
df.orderBy(F.col("defective_units").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

In [75]:
df.orderBy(F.col("plant_id").asc(), F.col("units_produced").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [77]:
df.orderBy(F.col("units_produced").desc()).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [78]:
df.orderBy(F.col("downtime_minutes").desc_nulls_last()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

In [79]:
df.groupBy("plant_id", "machine_id") \
    .agg(F.max("vibration_mm_s").alias("max_vibration")) \
    .orderBy(F.col("max_vibration").desc()) \
    .show()

+--------+----------+-------------+
|plant_id|machine_id|max_vibration|
+--------+----------+-------------+
|    PL03|     MC012|         8.56|
|    PL06|     MC030|         8.25|
|    PL06|     MC027|         7.85|
|    PL02|     MC010|          7.8|
|    PL04|     MC018|         7.79|
|    PL05|     MC024|         7.77|
|    PL03|     MC015|         7.76|
|    PL04|     MC016|         7.63|
|    PL02|     MC007|         7.61|
|    PL04|     MC017|         7.52|
|    PL05|     MC021|          7.5|
|    PL01|     MC004|         7.48|
|    PL05|     MC025|         7.46|
|    PL04|     MC019|         7.45|
|    PL06|     MC028|         7.43|
|    PL05|     MC022|          7.4|
|    PL03|     MC014|         7.37|
|    PL02|     MC006|         7.36|
|    PL02|     MC008|         7.32|
|    PL05|     MC023|         7.29|
+--------+----------+-------------+
only showing top 20 rows


8.Aggregation


In [80]:
from pyspark.sql import functions as F

In [81]:
df.select(F.sum("units_planned").alias("total_units_planned")).show()

+-------------------+
|total_units_planned|
+-------------------+
|           87094621|
+-------------------+



In [82]:
df.select(F.sum("units_produced").alias("total_units_produced")).show()

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+



In [83]:
df.select(F.sum("units_produced").alias("total_units_produced")).show()

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+



In [84]:
df.select(F.sum("defective_units").alias("total_defective_units")).show()

+---------------------+
|total_defective_units|
+---------------------+
|              2030244|
+---------------------+



In [85]:
df.select(F.avg("units_produced").alias("avg_units_produced")).show()

+------------------+
|avg_units_produced|
+------------------+
| 674.3990666666666|
+------------------+



In [86]:
df.select(F.max("units_produced").alias("max_production")).show()

+--------------+
|max_production|
+--------------+
|          1245|
+--------------+



In [87]:
df.select(F.min("units_produced").alias("min_production")).show()

+--------------+
|min_production|
+--------------+
|           205|
+--------------+



In [88]:
df.select(F.avg("downtime_minutes").alias("avg_downtime")).show()

+-----------------+
|     avg_downtime|
+-----------------+
|33.93149474428351|
+-----------------+



In [89]:
df.select(F.sum("energy_kwh").alias("total_energy_kwh")).show()

+--------------------+
|    total_energy_kwh|
+--------------------+
|2.8368014080000117E7|
+--------------------+



In [90]:
df.select(F.avg("temperature_c").alias("avg_temperature")).show()

+------------------+
|   avg_temperature|
+------------------+
|63.021532392167266|
+------------------+



In [91]:
df.select(F.avg("vibration_mm_s").alias("avg_vibration")).show()

+-----------------+
|    avg_vibration|
+-----------------+
|3.398737691604917|
+-----------------+



9.groupBy()

In [92]:
from pyspark.sql import functions as F

In [93]:
df.groupBy("plant_id") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy(F.desc("total_production")) \
    .show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+



In [94]:
df.groupBy("product_family") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy(F.desc("total_production")) \
    .show()

+--------------+----------------+
|product_family|total_production|
+--------------+----------------+
|        Valves|        20433082|
|         Pumps|        20208739|
|   Controllers|        20168572|
|        Motors|        20117495|
+--------------+----------------+



In [95]:
df.groupBy("plant_id") \
    .agg(F.sum("defective_units").alias("total_defects")) \
    .orderBy(F.desc("total_defects")) \
    .show()

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+



In [96]:
df.groupBy("shift") \
    .agg(F.round(F.avg("downtime_minutes"), 2).alias("avg_downtime")) \
    .orderBy(F.desc("avg_downtime")) \
    .show()

+-------+------------+
|  shift|avg_downtime|
+-------+------------+
|Evening|       34.22|
|  Night|       33.95|
|Morning|       33.62|
+-------+------------+



In [97]:
df.groupBy("product_family") \
    .agg(F.round(F.avg("energy_kwh"), 2).alias("avg_energy_kwh")) \
    .orderBy(F.desc("avg_energy_kwh")) \
    .show()

+--------------+--------------+
|product_family|avg_energy_kwh|
+--------------+--------------+
|        Motors|        237.22|
|         Pumps|        236.89|
|        Valves|        236.35|
|   Controllers|        235.15|
+--------------+--------------+



In [98]:
df.groupBy("quality_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+--------------+-----+
|quality_status|count|
+--------------+-----+
|          Pass|74328|
|        Rework|22928|
|        Reject|22744|
+--------------+-----+



In [99]:
df.groupBy("plant_id", "shift") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy("plant_id", F.desc("total_production")) \
    .show(20)

+--------+-------+----------------+
|plant_id|  shift|total_production|
+--------+-------+----------------+
|    PL01|  Night|         4562612|
|    PL01|Evening|         4477769|
|    PL01|Morning|         4447851|
|    PL02|Morning|         4561068|
|    PL02|  Night|         4558467|
|    PL02|Evening|         4426917|
|    PL03|  Night|         4599608|
|    PL03|Evening|         4487211|
|    PL03|Morning|         4460950|
|    PL04|Evening|         4540617|
|    PL04|Morning|         4497099|
|    PL04|  Night|         4492219|
|    PL05|Evening|         4475101|
|    PL05|Morning|         4431446|
|    PL05|  Night|         4416616|
|    PL06|  Night|         4604432|
|    PL06|Evening|         4458798|
|    PL06|Morning|         4429107|
+--------+-------+----------------+



In [100]:
df.groupBy("plant_id", "product_family") \
    .agg(F.sum("defective_units").alias("total_defects")) \
    .orderBy("plant_id", F.desc("total_defects")) \
    .show(30)

+--------+--------------+-------------+
|plant_id|product_family|total_defects|
+--------+--------------+-------------+
|    PL01|         Pumps|        87103|
|    PL01|        Valves|        84874|
|    PL01|        Motors|        84174|
|    PL01|   Controllers|        83569|
|    PL02|   Controllers|        87120|
|    PL02|        Motors|        84560|
|    PL02|         Pumps|        84491|
|    PL02|        Valves|        83476|
|    PL03|        Valves|        85333|
|    PL03|        Motors|        84926|
|    PL03|         Pumps|        84803|
|    PL03|   Controllers|        84202|
|    PL04|        Valves|        87084|
|    PL04|         Pumps|        83976|
|    PL04|        Motors|        83897|
|    PL04|   Controllers|        83161|
|    PL05|        Motors|        84331|
|    PL05|         Pumps|        83607|
|    PL05|   Controllers|        83589|
|    PL05|        Valves|        82975|
|    PL06|   Controllers|        86140|
|    PL06|        Valves|        85570|


In [101]:
df.groupBy("machine_id") \
    .agg(F.round(F.avg("temperature_c"), 2).alias("avg_temperature")) \
    .orderBy(F.desc("avg_temperature")) \
    .show(30)

+----------+---------------+
|machine_id|avg_temperature|
+----------+---------------+
|     MC019|          63.27|
|     MC020|          63.22|
|     MC009|          63.13|
|     MC029|          63.13|
|     MC014|          63.11|
|     MC023|          63.11|
|     MC027|           63.1|
|     MC022|          63.09|
|     MC015|          63.09|
|     MC017|          63.08|
|     MC001|          63.08|
|     MC011|          63.06|
|     MC007|          63.04|
|     MC016|          63.04|
|     MC026|          63.03|
|     MC021|          63.01|
|     MC005|          62.98|
|     MC028|          62.98|
|     MC030|          62.97|
|     MC024|          62.96|
|     MC013|          62.96|
|     MC008|          62.96|
|     MC004|          62.96|
|     MC012|          62.95|
|     MC002|          62.94|
|     MC006|          62.92|
|     MC025|           62.9|
|     MC003|          62.88|
|     MC010|          62.88|
|     MC018|          62.82|
+----------+---------------+



In [102]:
df.groupBy("supervisor") \
    .agg(F.sum("units_produced").alias("total_output")) \
    .orderBy(F.desc("total_output")) \
    .show()

+----------+------------+
|supervisor|total_output|
+----------+------------+
|     Farah|    10276986|
|    Harini|    10215803|
|     Deepa|    10130817|
|    Bharat|    10122430|
|     Anita|    10089966|
|    Charan|    10037116|
|     Gopal|    10031531|
|     Irfan|    10023239|
+----------+------------+



10.Joining DataFrames

In [103]:
from google.colab import files

uploaded = files.upload()


Saving plant_master.csv to plant_master.csv


In [104]:
from google.colab import files

uploaded = files.upload()

Saving machine_master.csv to machine_master.csv


In [105]:
plant_df = spark.read.csv("plant_master.csv", header=True, inferSchema=True)

machine_df = spark.read.csv("machine_master.csv", header=True, inferSchema=True)

In [106]:
prod_plant_df = df.join(plant_df, on="plant_id", how="inner")

prod_plant_df.show()
prod_plant_df.count()

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|    PL03|    PR0000001|     2026-08-07|     MC011|     OP0275|Evening|     MTR-100|    

120000

In [108]:
prod_plant_df.select(
    "production_id", "plant_name", "machine_id", "units_produced"
).show()

+-------------+---------------+----------+--------------+
|production_id|     plant_name|machine_id|units_produced|
+-------------+---------------+----------+--------------+
|    PR0000001|  Chennai Plant|     MC011|           993|
|    PR0000002|    Noida Plant|     MC019|           836|
|    PR0000003|Hyderabad Plant|     MC002|           380|
|    PR0000004|  Chennai Plant|     MC013|           720|
|    PR0000005|    Noida Plant|     MC020|           296|
|    PR0000006|Hyderabad Plant|     MC002|           795|
|    PR0000007|     Pune Plant|     MC007|           955|
|    PR0000008|     Pune Plant|     MC009|           895|
|    PR0000009|  Chennai Plant|     MC015|           289|
|    PR0000010|     Pune Plant|     MC006|           527|
|    PR0000011|  Chennai Plant|     MC012|           512|
|    PR0000012|  Kolkata Plant|     MC030|           968|
|    PR0000013|  Kolkata Plant|     MC027|           867|
|    PR0000014|  Chennai Plant|     MC013|           456|
|    PR0000015

In [109]:
machine_slim_df = machine_df.drop("plant_id")

prod_machine_df = df.join(machine_slim_df, on="machine_id", how="inner")

prod_machine_df.select(
    "production_id", "machine_id", "machine_type", "capacity_per_hour", "units_produced"
).show()

+-------------+----------+------------+-----------------+--------------+
|production_id|machine_id|machine_type|capacity_per_hour|units_produced|
+-------------+----------+------------+-----------------+--------------+
|    PR0000001|     MC011|         CNC|              100|           993|
|    PR0000002|     MC019|     Welding|               80|           836|
|    PR0000003|     MC002|    Assembly|               80|           380|
|    PR0000004|     MC013|     Casting|              180|           720|
|    PR0000005|     MC020|     Testing|              120|           296|
|    PR0000006|     MC002|    Assembly|               80|           795|
|    PR0000007|     MC007|    Assembly|              100|           955|
|    PR0000008|     MC009|     Welding|               80|           895|
|    PR0000009|     MC015|     Testing|              180|           289|
|    PR0000010|     MC006|         CNC|              100|           527|
|    PR0000011|     MC012|    Assembly|            

In [110]:
full_df = df \
    .join(plant_df, on="plant_id", how="inner") \
    .join(machine_slim_df, on="machine_id", how="inner")

full_df.printSchema()
full_df.count()

root
 |-- machine_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- year_opened: integer (nullable = true)
 |-- plant_manager: string (nullab

120000

In [111]:
full_df.groupBy("city") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy(F.desc("total_production")) \
    .show()

+---------+----------------+
|     city|total_production|
+---------+----------------+
|  Chennai|        13547769|
|     Pune|        13546452|
|    Noida|        13529935|
|  Kolkata|        13492337|
|Hyderabad|        13488232|
|Ahmedabad|        13323163|
+---------+----------------+



In [112]:
full_df.groupBy("machine_type") \
    .agg(F.round(F.avg("downtime_minutes"), 2).alias("avg_downtime")) \
    .orderBy(F.desc("avg_downtime")) \
    .show()

+------------+------------+
|machine_type|avg_downtime|
+------------+------------+
|     Casting|       34.17|
|     Welding|        34.0|
|    Assembly|       33.97|
|     Testing|       33.88|
|         CNC|       33.65|
+------------+------------+



In [113]:
full_df.groupBy("region") \
    .agg(F.sum("defective_units").alias("total_defects")) \
    .orderBy(F.desc("total_defects")) \
    .show()

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  West|       674149|
|  East|       338993|
| North|       338118|
+------+-------------+



In [114]:
no_machine_df = df.join(machine_df, on="machine_id", how="left_anti")

no_machine_df.show()
no_machine_df.count()

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



0

In [115]:
left_df = df.join(machine_slim_df, on="machine_id", how="left")

unmatched_df = left_df.filter(F.col("machine_type").isNull())

unmatched_df.show()
unmatched_df.count()

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+------------+-----------------+--------------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+-------

0

In [117]:
full_df.groupBy("plant_name") \
    .agg(F.round(F.sum("energy_kwh"), 2).alias("total_energy_kwh")) \
    .orderBy(F.desc("total_energy_kwh")) \
    .show()

+---------------+----------------+
|     plant_name|total_energy_kwh|
+---------------+----------------+
|     Pune Plant|      4757984.95|
|    Noida Plant|      4746238.58|
|  Chennai Plant|      4742748.17|
|Hyderabad Plant|      4725471.19|
|  Kolkata Plant|      4723180.16|
|Ahmedabad Plant|      4672391.03|
+---------------+----------------+



11.Updating data


In [118]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_upd = df

In [119]:
df_upd = df_upd.withColumn(
    "quality_status",
    F.when(F.col("quality_status") == "Reject", "Failed").otherwise(F.col("quality_status"))
)

df_upd.groupBy("quality_status").count().show()

+--------------+-----+
|quality_status|count|
+--------------+-----+
|        Rework|22928|
|        Failed|22744|
|          Pass|74328|
+--------------+-----+



In [120]:
df_upd = df_upd.withColumn(
    "quality_status",
    F.when(F.col("quality_status") == "Rework", "Requires Review").otherwise(F.col("quality_status"))
)

df_upd.groupBy("quality_status").count().show()

+---------------+-----+
| quality_status|count|
+---------------+-----+
|         Failed|22744|
|Requires Review|22928|
|           Pass|74328|
+---------------+-----+



In [121]:
df_upd = df_upd.withColumn(
    "units_planned",
    F.when(
        F.col("product_family") == "Motors",
        F.round(F.col("units_planned") * 1.05).cast("int")
    ).otherwise(F.col("units_planned"))
)

df_upd.groupBy("product_family").agg(F.sum("units_planned").alias("total_planned")).show()

+--------------+-------------+
|product_family|total_planned|
+--------------+-------------+
|         Pumps|     21744889|
|        Valves|     21986559|
|   Controllers|     21707690|
|        Motors|     22739080|
+--------------+-------------+



In [122]:
df_upd = df_upd.withColumn(
    "energy_kwh",
    F.when(
        F.col("shift") == "Night",
        F.round(F.col("energy_kwh") * 1.03, 2)
    ).otherwise(F.col("energy_kwh"))
)

df_upd.groupBy("shift").agg(F.round(F.sum("energy_kwh"), 2).alias("total_energy")).show()

+-------+------------+
|  shift|total_energy|
+-------+------------+
|Evening|  9410984.47|
|Morning|  9405926.53|
|  Night|   9837637.4|
+-------+------------+



In [123]:
df_upd = df_upd.fillna({"operator_id": "UNASSIGNED"})

df_upd.filter(F.col("operator_id") == "UNASSIGNED").count()

1524

In [124]:
df_upd = df_upd.withColumn(
    "supervisor",
    F.when(F.col("supervisor") == "Anita", "Anita Rao").otherwise(F.col("supervisor"))
)

df_upd.groupBy("supervisor").count().show()

+----------+-----+
|supervisor|count|
+----------+-----+
|     Farah|15238|
|    Harini|15080|
|     Irfan|14849|
|    Bharat|15112|
|    Charan|14930|
|     Gopal|14875|
|     Deepa|15027|
| Anita Rao|14889|
+----------+-----+



In [126]:
machine_window = Window.partitionBy("machine_id")

df_upd = df_upd.withColumn(
    "temperature_corrected",
    F.coalesce(
        F.col("temperature_c"),
        F.round(F.avg("temperature_c").over(machine_window), 2)
    )
)

df_upd.select("production_id", "machine_id", "temperature_c", "temperature_corrected") \
    .filter(F.col("temperature_c").isNull()) \
    .show()

+-------------+----------+-------------+---------------------+
|production_id|machine_id|temperature_c|temperature_corrected|
+-------------+----------+-------------+---------------------+
|    PR0000502|     MC001|         NULL|                63.08|
|    PR0000503|     MC001|         NULL|                63.08|
|    PR0001385|     MC001|         NULL|                63.08|
|    PR0004026|     MC001|         NULL|                63.08|
|    PR0005846|     MC001|         NULL|                63.08|
|    PR0006248|     MC001|         NULL|                63.08|
|    PR0006568|     MC001|         NULL|                63.08|
|    PR0006985|     MC001|         NULL|                63.08|
|    PR0007690|     MC001|         NULL|                63.08|
|    PR0008672|     MC001|         NULL|                63.08|
|    PR0013487|     MC001|         NULL|                63.08|
|    PR0014277|     MC001|         NULL|                63.08|
|    PR0014281|     MC001|         NULL|               

In [127]:
df_upd.filter(F.col("temperature_corrected").isNull()).count()


0

12.Null values and na Functions

In [128]:
from pyspark.sql import functions as F

In [129]:
df.filter(F.col("operator_id").isNull()).count()

1524

In [130]:
df.filter(F.col("temperature_c").isNull()).count()

2132

In [131]:
df.filter(F.col("vibration_mm_s").isNull()).count()

1788

In [132]:
df.filter(F.col("downtime_minutes").isNull()).count()

1177

In [133]:
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
]).show()

+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|            0|              0|       0|         0|       1524|    0|           0|             0|            0|             0|              0|            1177|         0|         2132|          1788|             0|               0|         0|
+-------------+-------------

In [134]:
df_dropped_all = df.na.drop()

df_dropped_all.count()

113531

In [135]:
df_temp_dropped = df.na.drop(subset=["temperature_c"])

df_temp_dropped.count()

117868

In [136]:
df_downtime_filled = df.na.fill({"downtime_minutes": 0})

df_downtime_filled.filter(F.col("downtime_minutes").isNull()).count()

0

In [137]:
df_operator_filled = df.fillna({"operator_id": "UNKNOWN"})

df_operator_filled.filter(F.col("operator_id") == "UNKNOWN").count()

1524

In [138]:
avg_temp = df.select(F.round(F.avg("temperature_c"), 2)).first()[0]
print("Average temperature:", avg_temp)

df_temp_filled = df.na.fill({"temperature_c": avg_temp})

df_temp_filled.filter(F.col("temperature_c").isNull()).count()

Average temperature: 63.02


0

In [139]:
df_clean = df.na.fill({
    "operator_id": "UNKNOWN",
    "downtime_minutes": 0,
    "temperature_c": avg_temp
})

print("Original rows:           ", df.count())
print("After na.drop():         ", df_dropped_all.count())
print("After dropping temp only:", df_temp_dropped.count())
print("After filling (cleansed):", df_clean.count())

df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns
]).show()

Original rows:            120000
After na.drop():          113531
After dropping temp only: 117868
After filling (cleansed): 120000
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|            0|              0|       0|         0|          0|    0|           0|             0|            0|             0|             

In [140]:
df.filter(F.col("temperature_c").isNotNull()).count()

117868

In [141]:
df.select(
    "production_id",
    "downtime_minutes",
    F.coalesce(F.col("downtime_minutes"), F.lit(0.0)).alias("downtime_filled")
).filter(F.col("downtime_minutes").isNull()).show()

+-------------+----------------+---------------+
|production_id|downtime_minutes|downtime_filled|
+-------------+----------------+---------------+
|    PR0000179|            NULL|            0.0|
|    PR0000695|            NULL|            0.0|
|    PR0000727|            NULL|            0.0|
|    PR0001039|            NULL|            0.0|
|    PR0001162|            NULL|            0.0|
|    PR0001172|            NULL|            0.0|
|    PR0001275|            NULL|            0.0|
|    PR0001458|            NULL|            0.0|
|    PR0001581|            NULL|            0.0|
|    PR0001582|            NULL|            0.0|
|    PR0001586|            NULL|            0.0|
|    PR0001610|            NULL|            0.0|
|    PR0001698|            NULL|            0.0|
|    PR0001774|            NULL|            0.0|
|    PR0001941|            NULL|            0.0|
|    PR0002001|            NULL|            0.0|
|    PR0002228|            NULL|            0.0|
|    PR0002561|     

In [142]:
df.select(
    "production_id",
    F.coalesce(F.col("temperature_c"), F.lit(avg_temp)).alias("temperature_final")
).show()

+-------------+-----------------+
|production_id|temperature_final|
+-------------+-----------------+
|    PR0000001|             61.4|
|    PR0000002|            64.37|
|    PR0000003|            54.13|
|    PR0000004|            64.47|
|    PR0000005|            72.59|
|    PR0000006|            64.91|
|    PR0000007|            62.76|
|    PR0000008|             57.1|
|    PR0000009|            66.55|
|    PR0000010|            60.96|
|    PR0000011|            52.81|
|    PR0000012|            69.79|
|    PR0000013|            50.88|
|    PR0000014|             67.2|
|    PR0000015|            76.39|
|    PR0000016|            44.92|
|    PR0000017|            75.09|
|    PR0000018|            59.91|
|    PR0000019|            62.23|
|    PR0000020|            76.92|
+-------------+-----------------+
only showing top 20 rows


13.Built in functions


In [143]:
from pyspark.sql import functions as F

In [158]:
df_fn = df

In [160]:
df_fn = df_fn.withColumn("product_family_upper", F.upper("product_family"))

df_fn.select("product_family", "product_family_upper").show()

+--------------+--------------------+
|product_family|product_family_upper|
+--------------+--------------------+
|        Motors|              MOTORS|
|   Controllers|         CONTROLLERS|
|        Motors|              MOTORS|
|         Pumps|               PUMPS|
|         Pumps|               PUMPS|
|        Motors|              MOTORS|
|        Valves|              VALVES|
|        Valves|              VALVES|
|   Controllers|         CONTROLLERS|
|   Controllers|         CONTROLLERS|
|         Pumps|               PUMPS|
|   Controllers|         CONTROLLERS|
|   Controllers|         CONTROLLERS|
|        Valves|              VALVES|
|        Motors|              MOTORS|
|   Controllers|         CONTROLLERS|
|   Controllers|         CONTROLLERS|
|        Motors|              MOTORS|
|        Valves|              VALVES|
|         Pumps|               PUMPS|
+--------------+--------------------+
only showing top 20 rows


In [162]:
df_fn = df_fn.withColumn("supervisor_lower", F.lower("supervisor"))

df_fn.select("supervisor", "supervisor_lower").show()

+----------+----------------+
|supervisor|supervisor_lower|
+----------+----------------+
|    Charan|          charan|
|     Farah|           farah|
|     Deepa|           deepa|
|     Farah|           farah|
|     Deepa|           deepa|
|     Gopal|           gopal|
|     Gopal|           gopal|
|     Irfan|           irfan|
|     Anita|           anita|
|     Gopal|           gopal|
|    Bharat|          bharat|
|     Gopal|           gopal|
|     Deepa|           deepa|
|     Deepa|           deepa|
|     Deepa|           deepa|
|     Irfan|           irfan|
|     Gopal|           gopal|
|     Farah|           farah|
|     Deepa|           deepa|
|     Irfan|           irfan|
+----------+----------------+
only showing top 20 rows


In [164]:
df_fn = df_fn.withColumn("operator_id_trimmed", F.trim("operator_id"))

df_fn.select("operator_id", "operator_id_trimmed").show()

+-----------+-------------------+
|operator_id|operator_id_trimmed|
+-----------+-------------------+
|     OP0275|             OP0275|
|     OP0489|             OP0489|
|     OP0189|             OP0189|
|     OP0076|             OP0076|
|     OP0501|             OP0501|
|     OP0501|             OP0501|
|     OP0645|             OP0645|
|     OP0106|             OP0106|
|     OP0386|             OP0386|
|     OP0593|             OP0593|
|     OP0141|             OP0141|
|     OP0047|             OP0047|
|     OP0172|             OP0172|
|     OP0446|             OP0446|
|     OP0238|             OP0238|
|     OP0013|             OP0013|
|     OP0533|             OP0533|
|     OP0162|             OP0162|
|     OP0526|             OP0526|
|     OP0563|             OP0563|
+-----------+-------------------+
only showing top 20 rows


In [165]:
df_fn = df_fn.withColumn("product_code_length", F.length("product_code"))

df_fn.select("product_code", "product_code_length").show()

+------------+-------------------+
|product_code|product_code_length|
+------------+-------------------+
|     MTR-100|                  7|
|     CTL-330|                  7|
|     MTR-300|                  7|
|     PMP-210|                  7|
|     PMP-110|                  7|
|     MTR-300|                  7|
|     VLV-120|                  7|
|     VLV-320|                  7|
|     CTL-130|                  7|
|     CTL-230|                  7|
|     PMP-210|                  7|
|     CTL-330|                  7|
|     CTL-330|                  7|
|     VLV-220|                  7|
|     MTR-300|                  7|
|     CTL-330|                  7|
|     CTL-330|                  7|
|     MTR-100|                  7|
|     VLV-320|                  7|
|     PMP-310|                  7|
+------------+-------------------+
only showing top 20 rows


In [166]:
df_fn = df_fn.withColumn("plant_machine", F.concat_ws("-", "plant_id", "machine_id"))

df_fn.select("plant_id", "machine_id", "plant_machine").show()

+--------+----------+-------------+
|plant_id|machine_id|plant_machine|
+--------+----------+-------------+
|    PL03|     MC011|   PL03-MC011|
|    PL04|     MC019|   PL04-MC019|
|    PL01|     MC002|   PL01-MC002|
|    PL03|     MC013|   PL03-MC013|
|    PL04|     MC020|   PL04-MC020|
|    PL01|     MC002|   PL01-MC002|
|    PL02|     MC007|   PL02-MC007|
|    PL02|     MC009|   PL02-MC009|
|    PL03|     MC015|   PL03-MC015|
|    PL02|     MC006|   PL02-MC006|
|    PL03|     MC012|   PL03-MC012|
|    PL06|     MC030|   PL06-MC030|
|    PL06|     MC027|   PL06-MC027|
|    PL03|     MC013|   PL03-MC013|
|    PL02|     MC007|   PL02-MC007|
|    PL04|     MC020|   PL04-MC020|
|    PL03|     MC011|   PL03-MC011|
|    PL03|     MC012|   PL03-MC012|
|    PL01|     MC005|   PL01-MC005|
|    PL03|     MC014|   PL03-MC014|
+--------+----------+-------------+
only showing top 20 rows


In [167]:
df_fn = df_fn.withColumn("temperature_1dp", F.round("temperature_c", 1))

df_fn.select("temperature_c", "temperature_1dp").show()

+-------------+---------------+
|temperature_c|temperature_1dp|
+-------------+---------------+
|         61.4|           61.4|
|        64.37|           64.4|
|        54.13|           54.1|
|        64.47|           64.5|
|        72.59|           72.6|
|        64.91|           64.9|
|        62.76|           62.8|
|         57.1|           57.1|
|        66.55|           66.6|
|        60.96|           61.0|
|        52.81|           52.8|
|        69.79|           69.8|
|        50.88|           50.9|
|         67.2|           67.2|
|        76.39|           76.4|
|        44.92|           44.9|
|        75.09|           75.1|
|        59.91|           59.9|
|        62.23|           62.2|
|        76.92|           76.9|
+-------------+---------------+
only showing top 20 rows


In [168]:
df_fn = df_fn.withColumn(
    "unit_gap",
    F.abs(F.col("units_planned") - F.col("units_produced"))
)

df_fn.select("units_planned", "units_produced", "unit_gap").show()

+-------------+--------------+--------+
|units_planned|units_produced|unit_gap|
+-------------+--------------+--------+
|         1060|           993|      67|
|          809|           836|      27|
|          442|           380|      62|
|          787|           720|      67|
|          344|           296|      48|
|          958|           795|     163|
|         1165|           955|     210|
|          862|           895|      33|
|          313|           289|      24|
|          532|           527|       5|
|          577|           512|      65|
|          980|           968|      12|
|         1004|           867|     137|
|          448|           456|       8|
|         1086|          1042|      44|
|          870|           732|     138|
|         1152|           963|     189|
|          541|           469|      72|
|          381|           380|       1|
|          646|           573|      73|
+-------------+--------------+--------+
only showing top 20 rows


In [169]:
df_fn = df_fn.withColumn(
    "production_id_new",
    F.regexp_replace("production_id", "PR", "PROD-")
)

df_fn.select("production_id", "production_id_new").show()

+-------------+-----------------+
|production_id|production_id_new|
+-------------+-----------------+
|    PR0000001|     PROD-0000001|
|    PR0000002|     PROD-0000002|
|    PR0000003|     PROD-0000003|
|    PR0000004|     PROD-0000004|
|    PR0000005|     PROD-0000005|
|    PR0000006|     PROD-0000006|
|    PR0000007|     PROD-0000007|
|    PR0000008|     PROD-0000008|
|    PR0000009|     PROD-0000009|
|    PR0000010|     PROD-0000010|
|    PR0000011|     PROD-0000011|
|    PR0000012|     PROD-0000012|
|    PR0000013|     PROD-0000013|
|    PR0000014|     PROD-0000014|
|    PR0000015|     PROD-0000015|
|    PR0000016|     PROD-0000016|
|    PR0000017|     PROD-0000017|
|    PR0000018|     PROD-0000018|
|    PR0000019|     PROD-0000019|
|    PR0000020|     PROD-0000020|
+-------------+-----------------+
only showing top 20 rows


In [170]:
df_fn = df_fn.withColumn("code_prefix", F.substring("product_code", 1, 3)) \
             .withColumn("code_number", F.substring("product_code", 5, 3))

df_fn.select("product_code", "code_prefix", "code_number").show()

+------------+-----------+-----------+
|product_code|code_prefix|code_number|
+------------+-----------+-----------+
|     MTR-100|        MTR|        100|
|     CTL-330|        CTL|        330|
|     MTR-300|        MTR|        300|
|     PMP-210|        PMP|        210|
|     PMP-110|        PMP|        110|
|     MTR-300|        MTR|        300|
|     VLV-120|        VLV|        120|
|     VLV-320|        VLV|        320|
|     CTL-130|        CTL|        130|
|     CTL-230|        CTL|        230|
|     PMP-210|        PMP|        210|
|     CTL-330|        CTL|        330|
|     CTL-330|        CTL|        330|
|     VLV-220|        VLV|        220|
|     MTR-300|        MTR|        300|
|     CTL-330|        CTL|        330|
|     CTL-330|        CTL|        330|
|     MTR-100|        MTR|        100|
|     VLV-320|        VLV|        320|
|     PMP-310|        PMP|        310|
+------------+-----------+-----------+
only showing top 20 rows


In [171]:
df_fn = df_fn.withColumn(
    "temperature_final",
    F.coalesce(F.col("temperature_c"), F.lit(63.02))
)

df_fn.select("temperature_c", "temperature_final") \
    .filter(F.col("temperature_c").isNull()) \
    .show()

+-------------+-----------------+
|temperature_c|temperature_final|
+-------------+-----------------+
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
|         NULL|            63.02|
+-------------+-----------------+
only showing top 20 rows


14.when()/otherwise()

In [173]:
df_cat = df.withColumn(
    "production_efficiency",
    F.col("units_produced") / F.col("units_planned") * 100
)

df_cat = df_cat.withColumn(
    "efficiency_category",
    F.when(F.col("production_efficiency") >= 100, "Above Target")
     .when(F.col("production_efficiency") >= 95, "On Target")
     .when(F.col("production_efficiency") >= 85, "Below Target")
     .otherwise("Critical")
)

df_cat.select("production_id", "production_efficiency", "efficiency_category").show()

+-------------+---------------------+-------------------+
|production_id|production_efficiency|efficiency_category|
+-------------+---------------------+-------------------+
|    PR0000001|    93.67924528301886|       Below Target|
|    PR0000002|   103.33745364647713|       Above Target|
|    PR0000003|    85.97285067873304|       Below Target|
|    PR0000004|     91.4866581956798|       Below Target|
|    PR0000005|    86.04651162790698|       Below Target|
|    PR0000006|    82.98538622129436|           Critical|
|    PR0000007|    81.97424892703863|           Critical|
|    PR0000008|   103.82830626450117|       Above Target|
|    PR0000009|    92.33226837060703|       Below Target|
|    PR0000010|    99.06015037593986|          On Target|
|    PR0000011|    88.73483535528595|       Below Target|
|    PR0000012|    98.77551020408163|          On Target|
|    PR0000013|    86.35458167330677|       Below Target|
|    PR0000014|   101.78571428571428|       Above Target|
|    PR0000015

In [174]:
df_cat = df_cat.withColumn(
    "defect_category",
    F.when(F.col("defective_units") == 0, "Perfect")
     .when(F.col("defective_units").between(1, 10), "Normal")
     .when(F.col("defective_units").between(11, 25), "Attention")
     .otherwise("High Defect")
)

df_cat.select("production_id", "defective_units", "defect_category").show()

+-------------+---------------+---------------+
|production_id|defective_units|defect_category|
+-------------+---------------+---------------+
|    PR0000001|             30|    High Defect|
|    PR0000002|              0|        Perfect|
|    PR0000003|             17|      Attention|
|    PR0000004|              7|         Normal|
|    PR0000005|              7|         Normal|
|    PR0000006|             50|    High Defect|
|    PR0000007|             10|         Normal|
|    PR0000008|             51|    High Defect|
|    PR0000009|              8|         Normal|
|    PR0000010|             26|    High Defect|
|    PR0000011|             14|      Attention|
|    PR0000012|             23|      Attention|
|    PR0000013|             27|    High Defect|
|    PR0000014|              6|         Normal|
|    PR0000015|              4|         Normal|
|    PR0000016|              0|        Perfect|
|    PR0000017|             28|    High Defect|
|    PR0000018|             18|      Att

In [175]:
df_cat = df_cat.withColumn(
    "machine_condition",
    F.when((F.col("temperature_c") > 75) | (F.col("vibration_mm_s") > 6), "Inspect")
     .otherwise("Normal")
)

df_cat.select("production_id", "temperature_c", "vibration_mm_s", "machine_condition").show()

+-------------+-------------+--------------+-----------------+
|production_id|temperature_c|vibration_mm_s|machine_condition|
+-------------+-------------+--------------+-----------------+
|    PR0000001|         61.4|          3.88|           Normal|
|    PR0000002|        64.37|          2.28|           Normal|
|    PR0000003|        54.13|          4.66|           Normal|
|    PR0000004|        64.47|          NULL|           Normal|
|    PR0000005|        72.59|          3.21|           Normal|
|    PR0000006|        64.91|          2.77|           Normal|
|    PR0000007|        62.76|          2.89|           Normal|
|    PR0000008|         57.1|          1.34|           Normal|
|    PR0000009|        66.55|          3.07|           Normal|
|    PR0000010|        60.96|          3.48|           Normal|
|    PR0000011|        52.81|          3.46|           Normal|
|    PR0000012|        69.79|          3.43|           Normal|
|    PR0000013|        50.88|           5.7|           

In [176]:
df_cat.groupBy("efficiency_category").count().orderBy(F.desc("count")).show()

+-------------------+-----+
|efficiency_category|count|
+-------------------+-----+
|       Below Target|54619|
|          On Target|26825|
|       Above Target|21735|
|           Critical|16821|
+-------------------+-----+



In [177]:
df_cat.groupBy("defect_category").count().orderBy(F.desc("count")).show()

+---------------+-----+
|defect_category|count|
+---------------+-----+
|      Attention|45745|
|         Normal|34706|
|    High Defect|28058|
|        Perfect|11491|
+---------------+-----+



In [178]:
df_cat.groupBy("machine_condition").count().orderBy(F.desc("count")).show()

+-----------------+------+
|machine_condition| count|
+-----------------+------+
|           Normal|112250|
|          Inspect|  7750|
+-----------------+------+



15Complex JSON

In [179]:
from google.colab import files

uploaded = files.upload()

Saving machine_configurations.json to machine_configurations.json


In [180]:
from pyspark.sql import functions as F

machine_json = spark.read.option("multiline", "true").json("machine_configurations.json")

In [181]:
machine_json.printSchema()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)



In [182]:
machine_json.select("machine_id", "machine_type").show()

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Testing|
|     MC006|         CNC|
|     MC007|    Assembly|
|     MC008|     Casting|
|     MC009|     Welding|
|     MC010|     Testing|
|     MC011|         CNC|
|     MC012|    Assembly|
|     MC013|     Casting|
|     MC014|     Welding|
|     MC015|     Testing|
|     MC016|         CNC|
|     MC017|    Assembly|
|     MC018|     Casting|
|     MC019|     Welding|
|     MC020|     Testing|
+----------+------------+
only showing top 20 rows


In [183]:
machine_json.select(
    "machine_id",
    F.col("specifications.rated_power_kw").alias("rated_power_kw")
).show()

+----------+--------------+
|machine_id|rated_power_kw|
+----------+--------------+
|     MC001|            22|
|     MC002|            22|
|     MC003|            55|
|     MC004|            22|
|     MC005|            30|
|     MC006|            30|
|     MC007|            22|
|     MC008|            55|
|     MC009|            30|
|     MC010|            55|
|     MC011|            30|
|     MC012|            55|
|     MC013|            30|
|     MC014|            30|
|     MC015|            45|
|     MC016|            30|
|     MC017|            30|
|     MC018|            30|
|     MC019|            30|
|     MC020|            15|
+----------+--------------+
only showing top 20 rows


In [184]:
machine_json.select(
    "machine_id",
    F.col("specifications.max_temperature_c").alias("max_temperature_c")
).show()

+----------+-----------------+
|machine_id|max_temperature_c|
+----------+-----------------+
|     MC001|               85|
|     MC002|               85|
|     MC003|               90|
|     MC004|               85|
|     MC005|               85|
|     MC006|               85|
|     MC007|               80|
|     MC008|               75|
|     MC009|               85|
|     MC010|               90|
|     MC011|               90|
|     MC012|               80|
|     MC013|               90|
|     MC014|               80|
|     MC015|               90|
|     MC016|               75|
|     MC017|               85|
|     MC018|               85|
|     MC019|               75|
|     MC020|               80|
+----------+-----------------+
only showing top 20 rows


In [185]:
machine_json.select(
    "machine_id",
    F.col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s")
).show()

+----------+--------------------+
|machine_id|vibration_limit_mm_s|
+----------+--------------------+
|     MC001|                 5.0|
|     MC002|                 5.5|
|     MC003|                 4.5|
|     MC004|                 4.5|
|     MC005|                 4.5|
|     MC006|                 5.5|
|     MC007|                 4.5|
|     MC008|                 5.0|
|     MC009|                 5.5|
|     MC010|                 4.5|
|     MC011|                 6.0|
|     MC012|                 5.5|
|     MC013|                 5.0|
|     MC014|                 5.5|
|     MC015|                 4.5|
|     MC016|                 5.5|
|     MC017|                 4.5|
|     MC018|                 5.0|
|     MC019|                 5.0|
|     MC020|                 5.0|
+----------+--------------------+
only showing top 20 rows


In [186]:
machine_json.select(
    "machine_id",
    F.col("maintenance.frequency_days").alias("maintenance_frequency_days"),
    F.col("maintenance.last_service").alias("last_service")
).show()

+----------+--------------------------+------------+
|machine_id|maintenance_frequency_days|last_service|
+----------+--------------------------+------------+
|     MC001|                        60|  2026-01-14|
|     MC002|                        60|  2026-05-19|
|     MC003|                        45|  2026-08-28|
|     MC004|                        60|  2026-04-19|
|     MC005|                        30|  2026-05-09|
|     MC006|                        60|  2026-01-09|
|     MC007|                        30|  2026-06-26|
|     MC008|                        30|  2026-04-21|
|     MC009|                        30|  2026-08-13|
|     MC010|                        45|  2026-06-05|
|     MC011|                        30|  2026-04-04|
|     MC012|                        30|  2026-04-26|
|     MC013|                        60|  2026-08-03|
|     MC014|                        90|  2026-01-21|
|     MC015|                        30|  2026-09-02|
|     MC016|                        45|  2026-

In [187]:
machine_json.select(
    "machine_id",
    "sensors",
    F.size("sensors").alias("sensor_count"),
    F.col("sensors")[0]["sensor"].alias("first_sensor"),
    F.col("sensors.sensor").alias("sensor_names"),
    F.array_contains(F.col("sensors.sensor"), "temperature").alias("has_temperature_sensor")
).show()

+----------+--------------------+------------+------------+--------------------+----------------------+
|machine_id|             sensors|sensor_count|first_sensor|        sensor_names|has_temperature_sensor|
+----------+--------------------+------------+------------+--------------------+----------------------+
|     MC001|[{true, temperatu...|           3| temperature|[temperature, vib...|                  true|
|     MC002|[{true, temperatu...|           3| temperature|[temperature, vib...|                  true|
|     MC003|[{true, temperatu...|           3| temperature|[temperature, vib...|                  true|
|     MC004|[{true, temperatu...|           3| temperature|[temperature, vib...|                  true|
|     MC005|[{true, temperatu...|           3| temperature|[temperature, vib...|                  true|
|     MC006|[{true, temperatu...|           3| temperature|[temperature, vib...|                  true|
|     MC007|[{true, temperatu...|           3| temperature|[temp

In [188]:
exploded_df = machine_json.select(
    "machine_id",
    F.explode("sensors").alias("sensor_info")
)

exploded_df.show()

+----------+--------------------+
|machine_id|         sensor_info|
+----------+--------------------+
|     MC001|{true, temperatur...|
|     MC001|{true, vibration,...|
|     MC001| {true, energy, kWh}|
|     MC002|{true, temperatur...|
|     MC002|{true, vibration,...|
|     MC002| {true, energy, kWh}|
|     MC003|{true, temperatur...|
|     MC003|{true, vibration,...|
|     MC003| {true, energy, kWh}|
|     MC004|{true, temperatur...|
|     MC004|{true, vibration,...|
|     MC004| {true, energy, kWh}|
|     MC005|{true, temperatur...|
|     MC005|{true, vibration,...|
|     MC005| {true, energy, kWh}|
|     MC006|{true, temperatur...|
|     MC006|{true, vibration,...|
|     MC006| {true, energy, kWh}|
|     MC007|{true, temperatur...|
|     MC007|{true, vibration,...|
+----------+--------------------+
only showing top 20 rows


In [189]:
machine_sensor_df = machine_json.select(
    "machine_id",
    "machine_type",
    F.explode("sensors").alias("s")
).select(
    "machine_id",
    "machine_type",
    F.col("s.sensor").alias("sensor"),
    F.col("s.unit").alias("unit"),
    F.col("s.active").alias("active")
)

machine_sensor_df.show()
machine_sensor_df.count()

+----------+------------+-----------+----+------+
|machine_id|machine_type|     sensor|unit|active|
+----------+------------+-----------+----+------+
|     MC001|         CNC|temperature|   C|  true|
|     MC001|         CNC|  vibration|mm/s|  true|
|     MC001|         CNC|     energy| kWh|  true|
|     MC002|    Assembly|temperature|   C|  true|
|     MC002|    Assembly|  vibration|mm/s|  true|
|     MC002|    Assembly|     energy| kWh|  true|
|     MC003|     Casting|temperature|   C|  true|
|     MC003|     Casting|  vibration|mm/s|  true|
|     MC003|     Casting|     energy| kWh|  true|
|     MC004|     Welding|temperature|   C|  true|
|     MC004|     Welding|  vibration|mm/s|  true|
|     MC004|     Welding|     energy| kWh|  true|
|     MC005|     Testing|temperature|   C|  true|
|     MC005|     Testing|  vibration|mm/s|  true|
|     MC005|     Testing|     energy| kWh|  true|
|     MC006|         CNC|temperature|   C|  true|
|     MC006|         CNC|  vibration|mm/s|  true|


90

In [190]:
machine_sensor_df.filter(
    (F.col("sensor") == "temperature") & (F.col("active") == True)
).select("machine_id", "machine_type").show()

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Testing|
|     MC006|         CNC|
|     MC007|    Assembly|
|     MC008|     Casting|
|     MC009|     Welding|
|     MC010|     Testing|
|     MC011|         CNC|
|     MC012|    Assembly|
|     MC013|     Casting|
|     MC014|     Welding|
|     MC015|     Testing|
|     MC016|         CNC|
|     MC017|    Assembly|
|     MC018|     Casting|
|     MC019|     Welding|
|     MC020|     Testing|
+----------+------------+
only showing top 20 rows


16.Union and Union all

In [191]:
from google.colab import files

uploaded = files.upload()

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv


In [192]:
from pyspark.sql import functions as F

extra_df = spark.read.csv(
    "manufacturing_production_extra_12k.csv",
    header=True,
    inferSchema=True
)

In [193]:
main_count = df.count()
main_count

120000

In [194]:
extra_count = extra_df.count()
extra_count

12500

In [195]:
union_df = df.union(extra_df)
union_df.count()

132500

In [196]:
union_all_df = df.unionAll(extra_df)
union_all_df.count()

132500

In [202]:
print(main_count)
print(extra_count)
print(union_df.count())
print(union_all_df.count())
print( main_count + extra_count)

120000
12500
132500
132500
132500


In [199]:
dup_ids = union_df.groupBy("production_id") \
    .count() \
    .filter(F.col("count") > 1)

dup_ids.show(10)
dup_ids.count()

+-------------+-----+
|production_id|count|
+-------------+-----+
|    PR0000413|    2|
|    PR0000135|    2|
|    PR0000236|    2|
|    PR0000339|    2|
|    PR0000059|    2|
|    PR0000463|    2|
|    PR0000136|    2|
|    PR0000263|    2|
|    PR0000393|    2|
|    PR0000445|    2|
+-------------+-----+
only showing top 10 rows


500

In [200]:
d_all_cols = union_df.dropDuplicates()
d_by_id = union_df.dropDuplicates(["production_id"])

In [201]:
print(union_df.count())
print( d_all_cols.count())
print(d_by_id.count())

132500
132500
132000


In [203]:
by_name_df = df.unionByName(extra_df)
by_name_df.count()

132500

In [204]:
cols = extra_df.columns[:]
i, j = cols.index("plant_id"), cols.index("machine_id")
cols[i], cols[j] = cols[j], cols[i]

extra_swapped = extra_df.select(*cols)
print(extra_swapped.columns[:5])

['production_id', 'production_date', 'machine_id', 'plant_id', 'operator_id']


In [206]:

wrong_df = df.union(extra_swapped)
print(wrong_df.select("plant_id").distinct().count())

right_df = df.unionByName(extra_swapped)
print(right_df.select("plant_id").distinct().count())

36
6


17.Window Functions


In [207]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [208]:
w_plant_rank = Window.partitionBy("plant_id").orderBy(F.col("units_produced").desc())

In [209]:
ranked_df = df.withColumn("row_num", F.row_number().over(w_plant_rank))

In [210]:
ranked_df = df.withColumn("row_num", F.row_number().over(w_plant_rank))

In [212]:
top5_df = ranked_df.filter(F.col("row_num") <= 5)

top5_df.select("plant_id", "production_id", "units_produced", "row_num") \
    .orderBy("plant_id", "row_num") \
    .show(30)

+--------+-------------+--------------+-------+
|plant_id|production_id|units_produced|row_num|
+--------+-------------+--------------+-------+
|    PL01|    PR0073059|          1239|      1|
|    PL01|    PR0002678|          1238|      2|
|    PL01|    PR0005192|          1236|      3|
|    PL01|    PR0044516|          1236|      4|
|    PL01|    PR0056319|          1236|      5|
|    PL02|    PR0100473|          1245|      1|
|    PL02|    PR0010732|          1243|      2|
|    PL02|    PR0001765|          1242|      3|
|    PL02|    PR0027036|          1238|      4|
|    PL02|    PR0012982|          1237|      5|
|    PL03|    PR0082579|          1243|      1|
|    PL03|    PR0030298|          1239|      2|
|    PL03|    PR0117602|          1239|      3|
|    PL03|    PR0102987|          1236|      4|
|    PL03|    PR0021534|          1235|      5|
|    PL04|    PR0008535|          1245|      1|
|    PL04|    PR0087560|          1245|      2|
|    PL04|    PR0047939|          1243| 

In [214]:
machine_totals = df.groupBy("plant_id", "machine_id") \
    .agg(F.sum("units_produced").alias("total_output"))

w_machine_rank = Window.partitionBy("plant_id").orderBy(F.col("total_output").desc())

machine_totals.withColumn("machine_rank", F.row_number().over(w_machine_rank)) \
    .filter(F.col("machine_rank") <= 3) \
    .orderBy("plant_id", "machine_rank") \
    .show(18)

+--------+----------+------------+------------+
|plant_id|machine_id|total_output|machine_rank|
+--------+----------+------------+------------+
|    PL01|     MC004|     2722412|           1|
|    PL01|     MC002|     2713779|           2|
|    PL01|     MC001|     2691431|           3|
|    PL02|     MC009|     2760070|           1|
|    PL02|     MC007|     2713038|           2|
|    PL02|     MC008|     2705850|           3|
|    PL03|     MC012|     2750570|           1|
|    PL03|     MC015|     2739284|           2|
|    PL03|     MC014|     2735500|           3|
|    PL04|     MC016|     2750780|           1|
|    PL04|     MC019|     2735839|           2|
|    PL04|     MC017|     2726300|           3|
|    PL05|     MC024|     2737331|           1|
|    PL05|     MC023|     2718075|           2|
|    PL05|     MC022|     2635629|           3|
|    PL06|     MC026|     2723306|           1|
|    PL06|     MC027|     2718877|           2|
|    PL06|     MC030|     2701993|      

In [216]:
w_plant = Window.partitionBy("plant_id")

df.withColumn("plant_total_output", F.sum("units_produced").over(w_plant)) \
    .select("production_id", "plant_id", "units_produced", "plant_total_output") \
    .show()

+-------------+--------+--------------+------------------+
|production_id|plant_id|units_produced|plant_total_output|
+-------------+--------+--------------+------------------+
|    PR0000003|    PL01|           380|          13488232|
|    PR0000006|    PL01|           795|          13488232|
|    PR0000019|    PL01|           380|          13488232|
|    PR0000026|    PL01|           663|          13488232|
|    PR0000028|    PL01|          1042|          13488232|
|    PR0000030|    PL01|           359|          13488232|
|    PR0000031|    PL01|           776|          13488232|
|    PR0000039|    PL01|           265|          13488232|
|    PR0000040|    PL01|           306|          13488232|
|    PR0000050|    PL01|           908|          13488232|
|    PR0000055|    PL01|           478|          13488232|
|    PR0000057|    PL01|           325|          13488232|
|    PR0000063|    PL01|           665|          13488232|
|    PR0000080|    PL01|           821|          1348823

In [218]:
w_machine = Window.partitionBy("machine_id")

df.withColumn("machine_avg_output", F.round(F.avg("units_produced").over(w_machine), 2)) \
    .select("production_id", "machine_id", "units_produced", "machine_avg_output") \
    .show()

+-------------+----------+--------------+------------------+
|production_id|machine_id|units_produced|machine_avg_output|
+-------------+----------+--------------+------------------+
|    PR0000026|     MC001|           663|            671.51|
|    PR0000057|     MC001|           325|            671.51|
|    PR0000088|     MC001|           676|            671.51|
|    PR0000135|     MC001|           375|            671.51|
|    PR0000139|     MC001|           378|            671.51|
|    PR0000159|     MC001|           762|            671.51|
|    PR0000216|     MC001|           431|            671.51|
|    PR0000232|     MC001|          1030|            671.51|
|    PR0000243|     MC001|           464|            671.51|
|    PR0000279|     MC001|           616|            671.51|
|    PR0000344|     MC001|           433|            671.51|
|    PR0000354|     MC001|           305|            671.51|
|    PR0000410|     MC001|           991|            671.51|
|    PR0000415|     MC00

In [219]:
w_cumulative = Window.partitionBy("machine_id") \
    .orderBy("production_date", "production_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative_df = df.withColumn("cumulative_output", F.sum("units_produced").over(w_cumulative))

cumulative_df.filter(F.col("machine_id") == "MC001") \
    .select("machine_id", "production_date", "production_id", "units_produced", "cumulative_output") \
    .orderBy("production_date", "production_id") \
    .show()

+----------+---------------+-------------+--------------+-----------------+
|machine_id|production_date|production_id|units_produced|cumulative_output|
+----------+---------------+-------------+--------------+-----------------+
|     MC001|     2026-01-01|    PR0002356|           437|              437|
|     MC001|     2026-01-01|    PR0030520|           997|             1434|
|     MC001|     2026-01-01|    PR0042532|           822|             2256|
|     MC001|     2026-01-01|    PR0050348|           817|             3073|
|     MC001|     2026-01-01|    PR0057063|           784|             3857|
|     MC001|     2026-01-01|    PR0069086|           472|             4329|
|     MC001|     2026-01-01|    PR0072621|          1078|             5407|
|     MC001|     2026-01-01|    PR0082917|          1027|             6434|
|     MC001|     2026-01-01|    PR0088125|          1023|             7457|
|     MC001|     2026-01-01|    PR0090359|           257|             7714|
|     MC001|

In [220]:
w_seq = Window.partitionBy("machine_id").orderBy("production_date", "production_id")

seq_df = df.withColumn("previous_output", F.lag("units_produced", 1).over(w_seq))

In [221]:
seq_df = seq_df.withColumn("next_output", F.lead("units_produced", 1).over(w_seq))


In [222]:
seq_df = seq_df.withColumn("change_from_previous", F.col("units_produced") - F.col("previous_output"))

seq_df.filter(F.col("machine_id") == "MC001") \
    .select("production_date", "production_id", "units_produced", "previous_output", "next_output", "change_from_previous") \
    .orderBy("production_date", "production_id") \
    .show()

+---------------+-------------+--------------+---------------+-----------+--------------------+
|production_date|production_id|units_produced|previous_output|next_output|change_from_previous|
+---------------+-------------+--------------+---------------+-----------+--------------------+
|     2026-01-01|    PR0002356|           437|           NULL|        997|                NULL|
|     2026-01-01|    PR0030520|           997|            437|        822|                 560|
|     2026-01-01|    PR0042532|           822|            997|        817|                -175|
|     2026-01-01|    PR0050348|           817|            822|        784|                  -5|
|     2026-01-01|    PR0057063|           784|            817|        472|                 -33|
|     2026-01-01|    PR0069086|           472|            784|       1078|                -312|
|     2026-01-01|    PR0072621|          1078|            472|       1027|                 606|
|     2026-01-01|    PR0082917|         

In [223]:
family_totals = df.groupBy("product_family") \
    .agg(F.sum("units_produced").alias("total_output"))

w_family = Window.orderBy(F.col("total_output").desc())

family_totals.withColumn("family_rank", F.rank().over(w_family)).show()

+--------------+------------+-----------+
|product_family|total_output|family_rank|
+--------------+------------+-----------+
|        Valves|    20433082|          1|
|         Pumps|    20208739|          2|
|   Controllers|    20168572|          3|
|        Motors|    20117495|          4|
+--------------+------------+-----------+



18.Date and Time Functions

In [224]:
df_dt = df.withColumn("production_date", F.to_date("production_date", "yyyy-MM-dd"))

df_dt.select("production_id", "production_date").printSchema()
df_dt.select("production_id", "production_date").show()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)

+-------------+---------------+
|production_id|production_date|
+-------------+---------------+
|    PR0000001|     2026-08-07|
|    PR0000002|     2026-08-03|
|    PR0000003|     2026-05-20|
|    PR0000004|     2026-01-15|
|    PR0000005|     2026-09-21|
|    PR0000006|     2026-09-03|
|    PR0000007|     2026-05-07|
|    PR0000008|     2026-08-17|
|    PR0000009|     2026-03-10|
|    PR0000010|     2026-02-13|
|    PR0000011|     2026-09-29|
|    PR0000012|     2026-07-10|
|    PR0000013|     2026-07-19|
|    PR0000014|     2026-05-05|
|    PR0000015|     2026-03-15|
|    PR0000016|     2026-03-03|
|    PR0000017|     2026-08-04|
|    PR0000018|     2026-01-30|
|    PR0000019|     2026-01-11|
|    PR0000020|     2026-02-18|
+-------------+---------------+
only showing top 20 rows


In [225]:
df_dt = df_dt \
    .withColumn("year", F.year("production_date")) \
    .withColumn("month", F.month("production_date")) \
    .withColumn("day", F.dayofmonth("production_date"))

df_dt.select("production_date", "year", "month", "day").show()

+---------------+----+-----+---+
|production_date|year|month|day|
+---------------+----+-----+---+
|     2026-08-07|2026|    8|  7|
|     2026-08-03|2026|    8|  3|
|     2026-05-20|2026|    5| 20|
|     2026-01-15|2026|    1| 15|
|     2026-09-21|2026|    9| 21|
|     2026-09-03|2026|    9|  3|
|     2026-05-07|2026|    5|  7|
|     2026-08-17|2026|    8| 17|
|     2026-03-10|2026|    3| 10|
|     2026-02-13|2026|    2| 13|
|     2026-09-29|2026|    9| 29|
|     2026-07-10|2026|    7| 10|
|     2026-07-19|2026|    7| 19|
|     2026-05-05|2026|    5|  5|
|     2026-03-15|2026|    3| 15|
|     2026-03-03|2026|    3|  3|
|     2026-08-04|2026|    8|  4|
|     2026-01-30|2026|    1| 30|
|     2026-01-11|2026|    1| 11|
|     2026-02-18|2026|    2| 18|
+---------------+----+-----+---+
only showing top 20 rows


In [226]:
df_dt = df_dt.withColumn("month_name", F.date_format("production_date", "MMMM"))

df_dt.select("production_date", "month_name").show()

+---------------+----------+
|production_date|month_name|
+---------------+----------+
|     2026-08-07|    August|
|     2026-08-03|    August|
|     2026-05-20|       May|
|     2026-01-15|   January|
|     2026-09-21| September|
|     2026-09-03| September|
|     2026-05-07|       May|
|     2026-08-17|    August|
|     2026-03-10|     March|
|     2026-02-13|  February|
|     2026-09-29| September|
|     2026-07-10|      July|
|     2026-07-19|      July|
|     2026-05-05|       May|
|     2026-03-15|     March|
|     2026-03-03|     March|
|     2026-08-04|    August|
|     2026-01-30|   January|
|     2026-01-11|   January|
|     2026-02-18|  February|
+---------------+----------+
only showing top 20 rows


In [227]:
df_dt = df_dt.withColumn("day_name", F.date_format("production_date", "EEEE"))

df_dt.select("production_date", "day_name").show()

+---------------+---------+
|production_date| day_name|
+---------------+---------+
|     2026-08-07|   Friday|
|     2026-08-03|   Monday|
|     2026-05-20|Wednesday|
|     2026-01-15| Thursday|
|     2026-09-21|   Monday|
|     2026-09-03| Thursday|
|     2026-05-07| Thursday|
|     2026-08-17|   Monday|
|     2026-03-10|  Tuesday|
|     2026-02-13|   Friday|
|     2026-09-29|  Tuesday|
|     2026-07-10|   Friday|
|     2026-07-19|   Sunday|
|     2026-05-05|  Tuesday|
|     2026-03-15|   Sunday|
|     2026-03-03|  Tuesday|
|     2026-08-04|  Tuesday|
|     2026-01-30|   Friday|
|     2026-01-11|   Sunday|
|     2026-02-18|Wednesday|
+---------------+---------+
only showing top 20 rows


In [228]:
df_dt = df_dt.withColumn("year_month", F.date_format("production_date", "yyyy-MM"))

df_dt.select("production_date", "year_month").show()

+---------------+----------+
|production_date|year_month|
+---------------+----------+
|     2026-08-07|   2026-08|
|     2026-08-03|   2026-08|
|     2026-05-20|   2026-05|
|     2026-01-15|   2026-01|
|     2026-09-21|   2026-09|
|     2026-09-03|   2026-09|
|     2026-05-07|   2026-05|
|     2026-08-17|   2026-08|
|     2026-03-10|   2026-03|
|     2026-02-13|   2026-02|
|     2026-09-29|   2026-09|
|     2026-07-10|   2026-07|
|     2026-07-19|   2026-07|
|     2026-05-05|   2026-05|
|     2026-03-15|   2026-03|
|     2026-03-03|   2026-03|
|     2026-08-04|   2026-08|
|     2026-01-30|   2026-01|
|     2026-01-11|   2026-01|
|     2026-02-18|   2026-02|
+---------------+----------+
only showing top 20 rows


In [229]:
df_dt.groupBy("year_month") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy("year_month") \
    .show()

+----------+----------------+
|year_month|total_production|
+----------+----------------+
|   2026-01|         8478059|
|   2026-02|         7810891|
|   2026-03|         8441121|
|   2026-04|         8211005|
|   2026-05|         8507542|
|   2026-06|         8400505|
|   2026-07|         8476416|
|   2026-08|         8529150|
|   2026-09|         8115221|
|   2026-10|         5957978|
+----------+----------------+



In [230]:
df_dt.groupBy("year_month") \
    .agg(F.sum("defective_units").alias("total_defects")) \
    .orderBy("year_month") \
    .show()

+----------+-------------+
|year_month|total_defects|
+----------+-------------+
|   2026-01|       211741|
|   2026-02|       197462|
|   2026-03|       212036|
|   2026-04|       208707|
|   2026-05|       212049|
|   2026-06|       210381|
|   2026-07|       210482|
|   2026-08|       214807|
|   2026-09|       202990|
|   2026-10|       149589|
+----------+-------------+



In [231]:
df_dt.groupBy("year_month", "plant_id") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy("year_month", "plant_id") \
    .show()

+----------+--------+----------------+
|year_month|plant_id|total_production|
+----------+--------+----------------+
|   2026-01|    PL01|         1451436|
|   2026-01|    PL02|         1419252|
|   2026-01|    PL03|         1429811|
|   2026-01|    PL04|         1399488|
|   2026-01|    PL05|         1375887|
|   2026-01|    PL06|         1402185|
|   2026-02|    PL01|         1342273|
|   2026-02|    PL02|         1278581|
|   2026-02|    PL03|         1305531|
|   2026-02|    PL04|         1292557|
|   2026-02|    PL05|         1324271|
|   2026-02|    PL06|         1267678|
|   2026-03|    PL01|         1442140|
|   2026-03|    PL02|         1359740|
|   2026-03|    PL03|         1416484|
|   2026-03|    PL04|         1447603|
|   2026-03|    PL05|         1411435|
|   2026-03|    PL06|         1363719|
|   2026-04|    PL01|         1350450|
|   2026-04|    PL02|         1369684|
+----------+--------+----------------+
only showing top 20 rows


In [232]:
march_df = df_dt.filter(
    F.col("production_date").between(F.to_date(F.lit("2026-03-01")), F.to_date(F.lit("2026-03-31")))
)

march_df.count()
march_df.agg(F.sum("units_produced").alias("march_production")).show()

+----------------+
|march_production|
+----------------+
|         8441121|
+----------------+



In [233]:
df_dt = df_dt.withColumn("date_plus_7", F.date_add("production_date", 7))

df_dt.select("production_date", "date_plus_7").show()

+---------------+-----------+
|production_date|date_plus_7|
+---------------+-----------+
|     2026-08-07| 2026-08-14|
|     2026-08-03| 2026-08-10|
|     2026-05-20| 2026-05-27|
|     2026-01-15| 2026-01-22|
|     2026-09-21| 2026-09-28|
|     2026-09-03| 2026-09-10|
|     2026-05-07| 2026-05-14|
|     2026-08-17| 2026-08-24|
|     2026-03-10| 2026-03-17|
|     2026-02-13| 2026-02-20|
|     2026-09-29| 2026-10-06|
|     2026-07-10| 2026-07-17|
|     2026-07-19| 2026-07-26|
|     2026-05-05| 2026-05-12|
|     2026-03-15| 2026-03-22|
|     2026-03-03| 2026-03-10|
|     2026-08-04| 2026-08-11|
|     2026-01-30| 2026-02-06|
|     2026-01-11| 2026-01-18|
|     2026-02-18| 2026-02-25|
+---------------+-----------+
only showing top 20 rows


In [234]:
df_dt = df_dt.withColumn("date_minus_30", F.date_sub("production_date", 30))

df_dt.select("production_date", "date_minus_30").show()

+---------------+-------------+
|production_date|date_minus_30|
+---------------+-------------+
|     2026-08-07|   2026-07-08|
|     2026-08-03|   2026-07-04|
|     2026-05-20|   2026-04-20|
|     2026-01-15|   2025-12-16|
|     2026-09-21|   2026-08-22|
|     2026-09-03|   2026-08-04|
|     2026-05-07|   2026-04-07|
|     2026-08-17|   2026-07-18|
|     2026-03-10|   2026-02-08|
|     2026-02-13|   2026-01-14|
|     2026-09-29|   2026-08-30|
|     2026-07-10|   2026-06-10|
|     2026-07-19|   2026-06-19|
|     2026-05-05|   2026-04-05|
|     2026-03-15|   2026-02-13|
|     2026-03-03|   2026-02-01|
|     2026-08-04|   2026-07-05|
|     2026-01-30|   2025-12-31|
|     2026-01-11|   2025-12-12|
|     2026-02-18|   2026-01-19|
+---------------+-------------+
only showing top 20 rows


In [235]:
df_dt = df_dt.withColumn("days_since_production", F.datediff(F.current_date(), "production_date"))

df_dt.select("production_date", "days_since_production").show()

+---------------+---------------------+
|production_date|days_since_production|
+---------------+---------------------+
|     2026-08-07|                   77|
|     2026-08-03|                   81|
|     2026-05-20|                  156|
|     2026-01-15|                  281|
|     2026-09-21|                   32|
|     2026-09-03|                   50|
|     2026-05-07|                  169|
|     2026-08-17|                   67|
|     2026-03-10|                  227|
|     2026-02-13|                  252|
|     2026-09-29|                   24|
|     2026-07-10|                  105|
|     2026-07-19|                   96|
|     2026-05-05|                  171|
|     2026-03-15|                  222|
|     2026-03-03|                  234|
|     2026-08-04|                   80|
|     2026-01-30|                  266|
|     2026-01-11|                  285|
|     2026-02-18|                  247|
+---------------+---------------------+
only showing top 20 rows


In [236]:
df_dt.groupBy("year_month") \
    .agg(F.sum("units_produced").alias("total_production")) \
    .orderBy(F.desc("total_production")) \
    .limit(1) \
    .show()

+----------+----------------+
|year_month|total_production|
+----------+----------------+
|   2026-08|         8529150|
+----------+----------------+

